In [1]:
%cd /drive2/ryusejong/LFF
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
import json 
import time 
import re
import random
import numpy as np 
from tqdm.auto import tqdm
from util.utils import set_seed, read_data, save_result, get_answer_from_text, chat_huggingface, construct_conversation
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel


seed = 42
set_seed(seed)

/drive2/ryusejong/LFF


/drive2/ryusejong/miniconda3/envs/llm1/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Base & Output file
base_path = "output/zeroshot_CoT/GSM8K_Llama-3-8B-Instruct_zeroshot_CoT_train_512_seed42_portion0.1.jsonl"
base_file = read_data(base_path)

new_path = "output/LFF_v9/GSM8K_Llama-3-8B-Instruct_LFF_v9_6_train_512_seed42_portion0.1.jsonl"
#new_path = "output/zeroshot_CoT/GSM8K_Llama-3-8B-Instruct_zeroshot_CoT_test_512_seed42_portion0.1.jsonl"
new_file = read_data(new_path)

print(f"length of base_file: {len(base_file)}")
print(f"length of new_file: {len(new_file)}")

length of base_file: 747
length of new_file: 747


In [6]:
correct_correct = []
correct_incorrect = []
incorrect_correct = []
incorrect_incorrect = []

for i in tqdm(range(len(base_file))):
    true_answer = base_file[i]["answer"]
    base_pred_answer = base_file[i]["pred_ans"]
    new_pred_answer = new_file[i]["pred_ans2"] if "pred_ans2" in new_file[i] else new_file[i]["pred_ans1"]
    #new_pred_answer = new_file[i]["pred_ans"]

    if base_pred_answer == true_answer:
        if new_pred_answer == true_answer:
            correct_correct.append(new_file[i])
        else:
            correct_incorrect.append(new_file[i])
    else:
        if new_pred_answer == true_answer:
            incorrect_correct.append(new_file[i])
        else:
            incorrect_incorrect.append(new_file[i])

print(f"correct_correct: {len(correct_correct)}\nIndex: {[o['index'] for o in correct_correct]}\n")
print(f"correct_incorrect: {len(correct_incorrect)}\nIndex: {[o['index'] for o in correct_incorrect]}\n")
print(f"incorrect_correct: {len(incorrect_correct)}\nIndex: {[o['index'] for o in incorrect_correct]}\n")
print(f"incorrect_incorrect: {len(incorrect_incorrect)}\nIndex: {[o['index'] for o in incorrect_incorrect]}\n")
print(f"total num: {len(correct_correct) + len(correct_incorrect) + len(incorrect_correct) + len(incorrect_incorrect)}")

100%|██████████| 747/747 [00:00<00:00, 628010.64it/s]

correct_correct: 618
Index: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 20, 22, 23, 24, 25, 27, 28, 30, 31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 53, 55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 77, 78, 79, 81, 82, 83, 84, 86, 87, 88, 89, 90, 91, 93, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 113, 114, 115, 116, 117, 118, 119, 120, 122, 123, 125, 126, 127, 128, 130, 131, 133, 134, 135, 136, 139, 140, 142, 143, 144, 145, 146, 147, 149, 150, 152, 153, 154, 155, 156, 160, 161, 162, 163, 164, 165, 166, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 184, 185, 186, 187, 188, 190, 191, 192, 193, 194, 196, 197, 198, 199, 200, 202, 204, 206, 207, 209, 210, 211, 212, 214, 215, 216, 217, 218, 219, 220, 221, 223, 224, 225, 226, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 241, 242, 243, 244, 247, 248, 250, 251, 252, 253, 254

In [7]:
correct_correct = []
correct_incorrect = []
incorrect_correct = []
incorrect_incorrect = []

for i in tqdm(range(len(base_file))):
    true_answer = base_file[i]["answer"]
    base_pred_answer = base_file[i]["pred_ans"]
    #new_pred_answer = new_file[i]["pred_ans2"] if "pred_ans2" in new_file[i] else new_file[i]["pred_ans1"]
    new_pred_answer = new_file[i]["pred_ans"]

    if base_pred_answer == true_answer:
        if new_pred_answer == true_answer:
            correct_correct.append(new_file[i])
        else:
            correct_incorrect.append(new_file[i])
    else:
        if new_pred_answer == true_answer:
            incorrect_correct.append(new_file[i])
        else:
            incorrect_incorrect.append(new_file[i])

print(f"correct_correct: {len(correct_correct)}\nIndex: {[o['index'] for o in correct_correct]}\n")
print(f"correct_incorrect: {len(correct_incorrect)}\nIndex: {[o['index'] for o in correct_incorrect]}\n")
print(f"incorrect_correct: {len(incorrect_correct)}\nIndex: {[o['index'] for o in incorrect_correct]}\n")
print(f"incorrect_incorrect: {len(incorrect_incorrect)}\nIndex: {[o['index'] for o in incorrect_incorrect]}\n")
print(f"total num: {len(correct_correct) + len(correct_incorrect) + len(incorrect_correct) + len(incorrect_incorrect)}")

  0%|          | 0/747 [00:00<?, ?it/s]


KeyError: 'pred_ans'